# 01 · T1/T2の起点・ラベル・時系列分割

採用したLはconfigの一か所だけで管理します。窓は経過時間 `[L,90]` 日、Cは自発、Bは全経路の再聴です。追跡不足を陰性にしません。

T2はデータ内の最初の聴取が推薦。T1は各分割内の最初の推薦の時点で既知の曲です。同一分割の同じユーザー×曲を両タスクに入れません。最初の5秒ビンで経路が混在するペアは初接触タスクから除外します。

In [1]:
from pathlib import Path
import numpy as np
import polars as pl
from IPython.display import display, Image
from relisten import config
np.random.seed(config.SEED)
print("作業フォルダ:", config.ROOT)

作業フォルダ: /Users/macuser/dev/paper1


In [2]:
from relisten.labels import prepare_labels
summary = prepare_labels()
display(summary)

task,split,pairs,users,positive_rate_c,positive_rate_b,preliminary_pairs_approx,preliminary_positive_rate_c,rate_delta
str,str,u32,u32,f64,f64,i64,f64,f64
"""T2""","""train""",755797,3554,0.090176,0.45425,740000,0.085,0.005176
"""T2""","""validation""",283134,3856,0.105275,0.459479,270000,0.1,0.005275
"""T2""","""test""",967884,4819,0.096279,0.431502,940000,0.09,0.006279
"""T1""","""train""",418556,3114,0.192885,0.750633,410000,0.186,0.006885
"""T1""","""validation""",488857,3664,0.196401,0.742215,480000,0.191,0.005401
"""T1""","""test""",982716,4485,0.192999,0.687234,960000,0.187,0.005999


## 感度分析

活動条件0/10/30日、下限1/3/7/14日、追跡60/90日。活動条件は毎回同じ窓で計算し直します。全再聴オフセットがあるため、早期に再聴していても後期の再聴を見落としません。

In [3]:
sensitivity = pl.read_csv(config.RESULTS / "label_sensitivity.csv")
display(sensitivity.filter((pl.col("horizon_days")==config.HORIZON_DAYS) & (pl.col("min_active_days")==30)))

task,split,pairs,users,positive_rate_c,positive_rate_b,lower_days,horizon_days,min_active_days
str,str,i64,i64,f64,f64,i64,i64,i64
"""O2""","""test""",522653,4835,0.281567,0.353651,1,90,30
"""O2""","""train""",386372,3487,0.367366,0.443855,1,90,30
"""O2""","""validation""",159024,3754,0.279819,0.350299,1,90,30
"""T1""","""test""",982716,4485,0.192999,0.687234,1,90,30
"""T1""","""train""",418556,3114,0.192885,0.750633,1,90,30
…,…,…,…,…,…,…,…,…
"""T1""","""train""",406678,2969,0.177074,0.691864,14,90,30
"""T1""","""validation""",467256,3422,0.182455,0.696406,14,90,30
"""T2""","""test""",905038,4518,0.084515,0.374261,14,90,30


## 保存した表とリーク防止

`pairs.parquet` は未選択起点と全再聴時刻、`main_labels.parquet` は活動条件適用済みのT1/T2です。ラベル・追跡活動量は未来を用いる結果変数であり、予測特徴量ではありません。学習ペアの結果は149日目までに確定します。O2は自発初接触の記述比較専用です。

In [4]:
import json
display(json.loads((config.PROCESSED / "pair_provenance.json").read_text()))
display(pl.read_csv(config.RESULTS / "anchor_counts.csv"))

{'max_timestamp': 26000000,
 'ambiguous_first_contact_pairs_excluded': 509,
 'split_ranges': {'train': [30, 59],
  'validation': [150, 164],
  'test': [165, 210]},
 'window': 'elapsed seconds: [L*86400, H*86400]',
 'source_revision': 'dd6f3a19eef5866e346c3270e098baa641a44948'}

task,split,all_anchors,censored_anchors
str,str,i64,i64
"""O2""","""test""",612147,971
"""O2""","""train""",446511,0
"""O2""","""validation""",185295,0
"""T1""","""test""",1064545,300
"""T1""","""train""",437046,0
"""T1""","""validation""",523734,0
"""T2""","""test""",1100797,1013
"""T2""","""train""",848521,0
"""T2""","""validation""",325261,0
